# Phase 1: Football Data Exploration
In this notebook, we explore the StatsBomb Open Data to understand the structure of football event data, particularly focusing on shots.
We will:
1. Connect to StatsBomb.
2. Retrieve competitions and matches.
3. Extract events from a specific match.
4. Analyze the properties of a 'Shot' event.


In [ ]:
# Install statsbombpy if not already installed
# !pip install statsbombpy pandas

In [ ]:
import pandas as pd
from statsbombpy import sb
import warnings
warnings.filterwarnings('ignore') # Ignore open data warnings

## 1. Retrieve Competitions
First, let's look at what competitions are available for free in the StatsBomb open dataset.

In [ ]:
# Fetch all competitions
competitions = sb.competitions()

# Filter for the FIFA World Cup
world_cup_comps = competitions[competitions['competition_name'] == 'FIFA World Cup']
world_cup_comps[['competition_id', 'season_id', 'season_name', 'competition_gender']]

## 2. Retrieve Matches
Let's select the 2022 Men's FIFA World Cup (competition_id=43, season_id=106) and retrieve all its matches.

In [ ]:
# Fetch matches for 2022 World Cup
matches = sb.matches(competition_id=43, season_id=106)
print(f"Total matches in 2022 World Cup: {len(matches)}")
matches.head(3)[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']]

## 3. Retrieve Match Events
Events contain the granular play-by-play data (passes, shots, fouls, etc.). Let's fetch events for the World Cup Final (Argentina vs France).

In [ ]:
# Fetch events for the 2022 Final (Argentina vs France)
# We can find the match_id by filtering
final_match = matches[(matches['home_team'] == 'Argentina') & (matches['away_team'] == 'France')].iloc[0]
match_id = final_match['match_id']

print(f"Fetching events for match_id: {match_id} (Argentina vs France)")
events = sb.events(match_id=match_id)

print(f"Total events in the match: {len(events)}")
print("\nUnique Event Types:")
print(events['type'].unique())

## 4. Explore Shot Data
For our platform, we only care about shots. Let's filter the events dataset and look at the structure of a shot.

In [ ]:
# Filter to only 'Shot' events
shots = events[events['type'] == 'Shot']
print(f"Total shots in the match: {len(shots)}")

# Display the key columns related to shots
shot_columns = [col for col in shots.columns if 'shot' in col or col in ['minute', 'second', 'player', 'team', 'location', 'play_pattern']]
shots[shot_columns].head()

### Deep Dive into a Single Shot
Let's examine the data points available for a single shot to understand what features we can use for our xG model.

In [ ]:
# Get the first shot
sample_shot = shots.iloc[0]

print("--- Sample Shot Details ---")
print(f"Player: {sample_shot['player']} ({sample_shot['team']})")
print(f"Time: {sample_shot['minute']}:{sample_shot['second']:02d}")
print(f"Location (X, Y): {sample_shot['location']}")
print(f"Body Part: {sample_shot['shot_body_part']}")
print(f"Shot Type: {sample_shot['shot_type']}")
print(f"Play Pattern: {sample_shot['play_pattern']}")
print(f"Outcome: {sample_shot['shot_outcome']}")
print(f"StatsBomb xG: {sample_shot['shot_statsbomb_xg']}")